## Midterm Project: Parallel Data Processing System (Single-Node)

---

In [ ]:
# imports
import pandas as pd
import numpy as np
import time
from multiprocessing import Pool, cpu_count

### Preparations
> Chosen dataset: **Global Covid-19 Dataset** (Source: Kaggle)

In [ ]:
# load dataset
df = pd.read_csv("global_covid19_data.csv")

# Convert date column
df["date"] = pd.to_datetime(df["date"])

# Replace missing values
df = df.fillna(0)

print("Dataset shape:", df.shape)
df.head()



Dataset shape: (184787, 7)


,date,country,cumulative_total_cases,daily_new_cases,active_cases,cumulative_total_deaths,daily_new_deaths
0,2020-02-15,Afghanistan,0.0,0.0,0.0,0.0,0.0
1,2020-02-16,Afghanistan,0.0,0.0,0.0,0.0,0.0
2,2020-02-17,Afghanistan,0.0,0.0,0.0,0.0,0.0
3,2020-02-18,Afghanistan,0.0,0.0,0.0,0.0,0.0
4,2020-02-19,Afghanistan,0.0,0.0,0.0,0.0,0.0


---

### Data Operations (Filtering, Aggregation, and Sorting)
> Additional notes for the operations we applied:
> * Filtering: cases per country
> * Aggregation: total/average deaths per country
> * Sorting: top countries by cases

#### Sequential Version

In [ ]:
# Function for sequential
def sequential_operations(df):

    # Filtering: Philippines only
    philippines = df[df["country"] == "Philippines"]

    # Aggregation: average deaths per country
    avg_deaths = df.groupby("country")["daily_new_deaths"].mean()

    # Sorting: top countries by total cases
    top_cases = df.groupby("country")["cumulative_total_cases"].max().sort_values(ascending=False)

    return philippines, avg_deaths, top_cases

#### Measure Sequential Time

In [ ]:
# Measure Sequential Time
start = time.time()

philippines_seq, avg_deaths_seq, top_cases_seq = sequential_operations(df)

end = time.time()

seq_time = end - start

print("Sequential Time:", seq_time)

Sequential Time: 0.049536943435668945


#### Parallel Version

In [ ]:
# Define worker function inside
def worker(chunk):
    # Filtering: Philippines
    philippines = chunk[chunk["country"] == "Philippines"]
    # Aggregation: daily_new_deaths mean per country
    avg_deaths = chunk.groupby("country")["daily_new_deaths"].mean()
    # Cases per country for sorting
    cases = chunk.groupby("country")["cumulative_total_cases"].max()
    return philippines, avg_deaths, cases

# Function for parallel
def parallel_operations(df):

    num_workers = cpu_count()

    # Split dataframe
    chunks = np.array_split(df, num_workers)

    # Parallel pool
    with Pool(processes=num_workers) as pool:
        results = pool.map(worker, chunks)

    # Combine results

    philippines = pd.concat([r[0] for r in results])

    avg_deaths = (
        pd.concat([r[1] for r in results])
        .groupby(level=0)
        .mean()
    )

    top_cases = (
        pd.concat([r[2] for r in results])
        .groupby(level=0)
        .max()
        .sort_values(ascending=False)
    )

    return philippines, avg_deaths, top_cases


if __name__ == "__main__":
    start_time = time.time()
    philippines_par, avg_deaths_par, top_cases_par = parallel_operations(df)
    end_time = time.time()
    par_time = end_time - start_time
    print("Parallel Execution Time:", par_time)

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Parallel Execution Time: 0.2677457332611084


#### Measure Parallel Time

In [ ]:
# Measure parallel execution time
start_time = time.time()

philippines_par, avg_deaths_par, top_cases_par = parallel_operations(df)

end_time = time.time()

par_time = end_time - start_time
print(f"Parallel Execution Time: {par_time:.4f} seconds")

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Parallel Execution Time: 0.1438 seconds


#### Comparison Table

In [ ]:
comparison_table = pd.DataFrame({
    "Method": ["Sequential", "Parallel"],
    "Execution Time (seconds)": [seq_time, par_time]
})

comparison_table["Speedup"] = [1, seq_time / par_time]

print(comparison_table)

       Method  Execution Time (seconds)   Speedup
0  Sequential                  0.049537  1.000000
1    Parallel                  0.143803  0.344478
